# Naive Bayes (Bag of Words) — MBTI Binary Dimensions

This notebook trains four binary Multinomial Naive Bayes classifiers (one per MBTI dimension) using Bag of Words vectorization.

## Overview

**Objective:** Predict each of the four MBTI dimensions independently:
- Introversion (I) vs. Extraversion (E)
- Intuition (N) vs. Sensing (S)
- Thinking (T) vs. Feeling (F)
- Judging (J) vs. Perceiving (P)


## 1 - Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from imblearn.over_sampling import RandomOverSampler

print("Packages imported successfully!")

## 2 - Load Data

Load the preprocessed MBTI dataset. The data contains:
- **posts**: Concatenated text posts from each user
- **type**: 4-letter MBTI type (e.g., "INFJ", "ENTP")

We use a flexible path search to ensure the notebook works across different team member environments.

In [ ]:
print("Loading data...")
current_dir = Path.cwd()
csv_path = None
for parent in [current_dir] + list(current_dir.parents)[:3]:
    candidate = parent / 'data' / 'processed' / 'preprocessed_data.csv'
    if candidate.exists():
        csv_path = candidate
        break
if not csv_path:
    csv_path = Path(r"C:\Users\piete\Documents\ML_MBTI_project\data\processed\preprocessed_data.csv")

df = pd.read_csv(csv_path)
df = df.dropna(subset=['posts', 'type'])
print(f"Data loaded successfully! Shape: {df.shape}")

## 3 - Vectorization (Bag of Words)

- `stop_words='english'`: Remove common words like "the", "is", "and" that don't carry personality information

**Bag of Words (BoW) Approach:**- `max_features=2000`: Keep top 2000 most frequent words (reduces dimensionality)

- Represents text as word frequency counts**Parameters:**

- Each document becomes a vector where each dimension represents a word

- Value = how many times that word appears- Generally faster to compute

- Simple but effective for text classification- Complements TF-IDF approach for comparison

- For personality detection, word repetition patterns may be significant

**Why BoW instead of TF-IDF?**- BoW preserves raw frequency information

In [ ]:
print("Vectorizing text using Bag of Words (CountVectorizer)...")

# We use CountVectorizer instead of TfidfVectorizer
# max_features=2000 ensures we keep the model size manageable
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(df['posts'])
y = df['type'].values

print(f"Data Ready! Features shape: {X.shape}")

## 4 - Train & Evaluate Models

### 4.1 Naive Bayes Theory

**Multinomial Naive Bayes:**

Based on Bayes' Theorem:
$$P(y|X) = \frac{P(X|y) \cdot P(y)}{P(X)}$$

For text classification:
$$P(y|x_1, ..., x_n) \propto P(y) \prod_{i=1}^{n} P(x_i|y)$$

where:
- $y$ is the class (e.g., I or E)
- $x_i$ are the word features
- "Naive" assumption: features are conditionally independent given the class

**Why Multinomial NB for text?**
- Designed for discrete features (word counts)
- Fast training and prediction
- Works well with high-dimensional sparse data (text)
- Performs surprisingly well despite the "naive" independence assumption

**Regularization (alpha parameter):**
- `alpha=5.0`: Laplace smoothing parameter
- Prevents zero probabilities for unseen words
- Higher alpha = more smoothing = less overfitting
- Default is 1.0, we use 5.0 for stronger regularization

### 4.2 Training Strategy

We train four separate binary classifiers, one for each MBTI dimension.

**Class Imbalance Handling:**
- RandomOverSampler: Duplicates minority class samples to balance classes
- Critical for Naive Bayes which can be sensitive to imbalanced data
- Applied before train/test split to maintain realistic test distribution

**Train/Test Split:**
- 80% training, 20% testing
- Stratified split maintains class proportions
- Fixed random_state=42 for reproducibility

**Evaluation Metrics:**
- **Accuracy**: Overall correct predictions
- **F1-Score (Macro)**: Harmonic mean of precision and recall, averaged across classes
- **Confusion Matrix**: Detailed breakdown of predictions vs. actual labels
- **Overfitting Check**: Compare train vs. test accuracy (>5% gap indicates overfitting)

In [ ]:
print("Starting Naive Bayes (BoW) Training...")

dimensions = [('I', 'E'), ('N', 'S'), ('T', 'F'), ('J', 'P')]
results = {}
f1_scores = []

for trait1, trait2 in dimensions:
    print(f"\nProcessing {trait1} vs {trait2}")
    
    y_binary = np.array([1 if trait2 in label else 0 for label in y])
    
    ros = RandomOverSampler(random_state=42)
    X_resampled, y_resampled = ros.fit_resample(X, y_binary)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_resampled, y_resampled, test_size=0.20, random_state=42
    )
    
    # Train with alpha for regularization
    clf = MultinomialNB(alpha=5.0)
    clf.fit(X_train, y_train)
    
    # Evaluate on both train and test sets
    train_preds = clf.predict(X_train)
    train_acc = accuracy_score(y_train, train_preds)
    
    test_preds = clf.predict(X_test)
    test_acc = accuracy_score(y_test, test_preds)

    current_f1 = f1_score(y_test, test_preds, average='macro')
    f1_scores.append(current_f1) 
    
    conf_mat = confusion_matrix(y_test, test_preds)
    
    results[f"{trait1}/{trait2}"] = test_acc
    
    # Display results with overfitting detection
    print(f"   Train Accuracy: {train_acc:.2%}")
    print(f"   Test Accuracy:  {test_acc:.2%}")
    print(f"   Difference:     {(train_acc - test_acc):.2%}")
    if train_acc - test_acc > 0.05:
        print(f"   Possible overfitting detected!")

    print(f"   Macro F1-Score: {current_f1:.4f}")
    print(f"   Confusion Matrix:\n{conf_mat}")

# Summary statistics
total_accuracy = sum(results.values())
avg_accuracy = total_accuracy / len(results)
avg_f1 = sum(f1_scores) / len(f1_scores)
print(f"\nTotal Sum of Accuracies: {total_accuracy:.4f}")
print(f"Average Accuracy: {avg_accuracy:.2%}")
print(f"Average F1 Score: {avg_f1:.4f}")

## 5 - Visualize Results

Each bar represents the test accuracy for one binary classification task.

Visualize model performance across all four MBTI dimensions to identify:

- Which dimensions are easier/harder to predict- Overall model effectiveness
- Consistency across different personality traits

In [ ]:
print("Generating BoW Results Graph...")

plt.figure(figsize=(10, 6))
keys = list(results.keys())
values = list(results.values())

# Using a slightly different color palette to distinguish from TF-IDF
bars = plt.bar(keys, values, color=['#e67e22', '#2ecc71', '#e74c3c', '#9b59b6'])

plt.title('Naive Bayes Accuracy (Bag of Words)', fontsize=16, pad=20)
plt.xlabel('Personality Dimension Pair', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.ylim(0, 1.0) 

# Add percentages
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height,
             f'{height:.2%}', 
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.show()